# P13 · Order Flow, Market Impact y exposición
**QuantLab — Módulo 1 · ampliación** | Lecciones 1.7, 1.8 y 1.27

## 1. Objetivos
- Reconstruir un libro L2 didáctico a partir de un snapshot y actualizaciones absolutas.
- Distinguir OBI (estado), OFI (cambio del mejor bid/ask) y volumen agresor (trades).
- Medir la relación contemporánea entre OFI y precio, y separarla de la predicción futura.
- Medir respuesta del precio a operaciones agresivas y exposición long/short.

**Fuentes:** simulación reproducible, o CSV de cotizaciones de P06 del exchange elegido. El CSV REST solo proporciona cambios entre snapshots, no OFI exacto por evento. No se atribuyen cancelaciones ni ejecuciones a partir de cotizaciones aisladas.

**Referencia:** Cont, Kukanov y Stoikov (2014), *The Price Impact of Order Book Events*, [artículo](https://arxiv.org/abs/1011.6402). Su evidencia en acciones no garantiza resultados en BTC.

Las funciones están en **`src/quantlab/p13/order_flow.py`** y sus pruebas en **`tests/test_order_flow.py`** (`pytest` desde la raíz del proyecto). Los ejercicios están resueltos en la sección 10.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display
from pathlib import Path
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p13.order_flow import (add_future, exposure, features, impact_by_quartile, impact_experiment,
                                     manual_ofi, manual_ofi_series, ofi_regressions, prepare_quotes, replay,
                                     sign_rule, simulate_quotes, window_ofi)

DATA_MODE = 'synthetic'  # 'csv' para usar P06 o tu recolector L1
CSV_PATH = 'p06_quotes.csv'
EXCHANGE = 'kraken'
SYMBOL = 'BTC/EUR'
MAX_GAP_SECONDS = 2.0
EXPORT = False
plt.rcParams.update({'figure.figsize':(9,3.5),'axes.grid':True,'grid.alpha':.22,'font.size':10})

## 2. Reconstrucción de un libro de órdenes
En L2 conocemos la cantidad agregada por precio, no las órdenes individuales. Para este ejemplo los mensajes `size` son cantidades **absolutas**; cero elimina un nivel. Cada mensaje es atómico. Una secuencia perdida obliga a resincronizar desde un snapshot. Las etiquetas de evento son conocidas solo porque diseñamos el ejemplo.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los mensajes del libro son coherentes?</b>

`replay` se detiene con un error si:

- Falta un número de secuencia: hay que **resincronizar** desde un snapshot nuevo, nunca seguir con un libro incompleto.
- El lado no es `bid`/`ask`, la cantidad es negativa o el precio ≤ 0.
- Tras aplicar un mensaje, el mejor bid alcanza o supera al mejor ask (libro cruzado).

Además registra la profundidad total de cada lado, para ver qué ocurre fuera del primer nivel.

</div>

In [ ]:
print(inspect.getsource(replay))

snapshot={'bid':{99:2.,98:3.},'ask':{101:2.,102:4.}}
updates=[(1,'bid',99,3.,'alta bid'),(2,'ask',101,1.,'ejecución compra'),
         (3,'bid',99,1.,'cancelación bid'),(4,'ask',101,0.,'ejecución compra'),
         (5,'bid',100,1.5,'mejora bid'),(6,'ask',101,2.,'mejora ask')]
example=replay(snapshot,updates)
display(example)

## 3. OFI de primer nivel
Sea $b,a$ el mejor precio bid/ask y $q^b,q^a$ sus cantidades. La contribución de cada cambio es:

$$e_t=\mathbf{1}_{b_t\ge b_{t-1}}q^b_t-\mathbf{1}_{b_t\le b_{t-1}}q^b_{t-1}-\mathbf{1}_{a_t\le a_{t-1}}q^a_t+\mathbf{1}_{a_t\ge a_{t-1}}q^a_{t-1}$$

A precio constante equivale a cambio de cantidad bid menos cambio de cantidad ask. Si cambia un mejor precio, usamos las cantidades correspondientes al nivel que aparece o desaparece. OFI es la suma de contribuciones en la ventana (BTC). OBI = (cantidad bid − cantidad ask)/(suma de ambas). No son intercambiables.

In [ ]:
print(inspect.getsource(features))

example=features(example)
display(example.round(4))

## 4. Datos de investigación
La simulación genera cambios de cotización una vez por segundo; las variaciones de precio tienen un mecanismo artificial de persistencia de corta duración. No reproduce el matching engine real. En modo CSV seleccionamos únicamente BTC/EUR de un exchange y ordenamos por recepción UTC. No mezclamos observaciones de mercados distintos ni rellenamos huecos. Un salto superior al límite rompe la continuidad.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿las cotizaciones son utilizables?</b>

`prepare_quotes` filtra un único exchange y par, y se detiene con un error si faltan columnas, hay menos de 60 cotizaciones, horas vacías o duplicadas, precios o tamaños ≤ 0, o ask ≤ bid.

Un salto mayor que `MAX_GAP_SECONDS` abre un **segmento nuevo**: su primera fila no tiene OFI ni cambio de mid, porque no se calculan a través de un hueco.

</div>

In [ ]:
print(inspect.getsource(simulate_quotes))
print(inspect.getsource(prepare_quotes))

raw=simulate_quotes(2400,73,EXCHANGE,SYMBOL) if DATA_MODE=='synthetic' else pd.read_csv(CSV_PATH) if DATA_MODE=='csv' else None
if raw is None: raise ValueError('DATA_MODE incorrecto')
x = prepare_quotes(raw, EXCHANGE, SYMBOL, MAX_GAP_SECONDS)
print(DATA_MODE,EXCHANGE,SYMBOL,len(x),'cotizaciones;',x.segment.nunique(),'segmentos')
display(x.head())

## 5. Visualizar estado y flujo
Una OBI positiva puede coexistir con un OFI negativo: hay más liquidez compradora en este instante, pero se está reduciendo. En el gráfico el OFI se representa en BTC por actualización, no como porcentaje.

In [ ]:
view=x.iloc[:100].set_index('received_at')
fig,ax=plt.subplots(2,1,sharex=True,figsize=(9,5),layout='constrained')
ax[0].plot(view.index,view.obi,color='#416c9b');ax[0].set(ylabel='OBI',title=f'Estado vs. flujo · {DATA_MODE} · primeras 100 cotizaciones')
ax[1].plot(view.index,view.ofi_event,color='#a66b39');ax[1].set(ylabel='OFI (BTC)',xlabel='UTC')
for a in ax:a.axhline(0,color='gray',lw=.7)
plt.show()

## 6. Relación contemporánea y predicción
Estimamos una regresión de cambio del mid (bps) sobre OFI (BTC). La primera usa cambios del mismo intervalo y **no es una señal operable**. Para predicción usamos solo el OFI conocido en t y el cambio entre t y t+1 actualización. En CSV irregular el horizonte no equivale a un segundo. Usamos un corte cronológico 65/35 y eliminamos el objetivo que cruza la frontera. HAC es una aproximación a la incertidumbre bajo dependencia temporal; no prueba causalidad.

In [ ]:
print(inspect.getsource(add_future))
print(inspect.getsource(ofi_regressions))

cut=int(len(x)*.65)
x = add_future(x, 1, cut)
result = ofi_regressions(x, cut, {'contemporáneo':'mid_change_bps','futuro':'future_bps'})
display(result.round(5))
fig,ax=plt.subplots(1,2,figsize=(9,3.6),layout='constrained')
for a,y,title in zip(ax,['mid_change_bps','future_bps'],['Mismo intervalo','Siguiente actualización']):
    d=x.iloc[cut:];a.scatter(d.ofi_event,d[y],s=8,alpha=.3,color='#416c9b');a.set(xlabel='OFI (BTC)',ylabel='Cambio mid (bps)',title=title)
fig.suptitle(f'Validación cronológica · {DATA_MODE}');plt.show()
print("Correlación entre el OFI y el SIGNO del cambio de mid del mismo intervalo:", round(x.ofi_event.corr(np.sign(x.mid_change_bps)), 3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La relación contemporánea es casi mecánica; la predictiva, mínima</b>

| Objetivo | Periodo | n | Pendiente (bps por BTC) | p-valor | R² |
|---|---|---|---|---|---|
| Contemporáneo | Desarrollo | 1.559 | 0,251 | ≈ 0 | **0,612** |
| Contemporáneo | Validación | 840 | 0,248 | ≈ 0 | **0,607** |
| Futuro (t + 1) | Desarrollo | 1.558 | 0,047 | ≈ 0 | 0,021 |
| Futuro (t + 1) | Validación | 839 | 0,038 | 0,001 | 0,014 |

- **¿Por qué un R² de 0,61 si los tamaños son aleatorios?** En la simulación, las cantidades son independientes del precio. Pero la fórmula del OFI suma o resta según si el mejor bid o ask **subió o bajó**: el OFI lleva dentro el signo del movimiento del precio. Su correlación con el signo del cambio de mid del mismo intervalo es de **0,85**. La relación contemporánea es en gran parte **mecánica**, no información sobre el futuro.
- **La relación futura existe, pero es diminuta:** R² de 0,01–0,02. Viene de la persistencia artificial de un paso (el 20 % del shock anterior se arrastra), no de los tamaños del libro.
- La pendiente se mantiene entre periodos, pero eso no la convierte en una señal operable (ver la regla).

</div>

### Comprobar una regla fuera de muestra
No reestimamos el signo de negociación en validación. Usamos la pendiente futura de desarrollo y el signo de OFI. El retorno mid teórico no es ejecutable. Restamos un coste hipotético total de 20 bps a cada operación; las ventas cortas requerirían infraestructura de préstamo o derivados. No es un backtest de cartera real.

In [ ]:
print(inspect.getsource(sign_rule))

train=result[(result.target=='futuro')&(result.period=='desarrollo')]
if train.empty:
    print('Sin datos suficientes para estimar la regla')
else:
    rule = sign_rule(x, cut, train.slope_bps_per_btc.iloc[0], 'future_bps', 20.)
    if rule['trades'] == 0: print('No hay operaciones evaluables')
    else: print('Operaciones:',rule['trades'],'| Bruto medio:',round(rule['gross_bps'],4),'bps | Neto:',round(rule['net_bps'],4),
                'bps | Acierto:', f"{rule['hit_rate']:.1%}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Acertar un 56 % de las veces no paga 20 bps</b>

- La regla opera en el sentido del OFI en las 839 actualizaciones de validación y acierta el signo en un **55,7 %** de los casos.
- Pero el movimiento medio capturado es de **0,04 bps**, frente a un coste de 20 bps: el neto es de **−19,96 bps** por operación.
- El mid se mueve unos 2 € por segundo (≈ 0,3 bps): ningún acierto sobre movimientos tan pequeños cubre un spread más comisión.

</div>

## 7. Market Impact: volumen y respuesta temporal
La respuesta firmada de un trade agresor es el cambio del mid desde antes del trade, multiplicado por el signo del agresor. El impacto inmediato puede revertir parcialmente; una respuesta a 10 segundos no demuestra impacto permanente. Para atribuir causalidad hay que controlar noticias, flujo simultáneo y selección de órdenes.

Este laboratorio controlado inyecta un efecto temporal que decae y un componente persistente. **No son datos empíricos ni parámetros estimados.** Con cotizaciones P06 no podemos calcular la respuesta a trades: se requiere un flujo de trades sincronizado con cotizaciones antes y después.

In [ ]:
print(inspect.getsource(impact_experiment))

events, horizons = impact_experiment(600, 83)
response = events[[f'response_{h}s_bps' for h in horizons]].to_numpy()
qty = events.qty.to_numpy()
fig,ax=plt.subplots(figsize=(8,3.5),layout='constrained')
for name,mask in [('Tamaño bajo',qty<=np.median(qty)),('Tamaño alto',qty>np.median(qty))]:
    ax.plot(horizons,response[mask].mean(axis=0),marker='o',label=name)
ax.set(xlabel='Segundos tras el trade',ylabel='Respuesta firmada (bps)',title='Respuesta temporal · experimento artificial');ax.legend();plt.show()
display(pd.DataFrame({'horizon_s':horizons,'mean_response_bps':response.mean(axis=0)}).round(3))

## 8. Lección 1.27: posiciones y exposición
Para instrumentos lineales, $PnL=q(P_t-P_0)$ y $N=qP_t$. La exposición bruta suma nominales absolutos, la neta suma nominales con signo. Para comparar con el patrimonio dividimos por equity positivo. No se modelan márgenes ni liquidaciones.

In [ ]:
print(inspect.getsource(exposure))

positions=pd.DataFrame({'instrument':['BTC spot','BTC perpetual'], 'quantity':[.1,-.1], 'entry_price':[60000,60000], 'current_price':[63000,63100]})
ledger,exposures=exposure(positions,7000)
display(ledger);display(pd.Series(exposures).to_frame('valor'))
fig,ax=plt.subplots(figsize=(8,3),layout='constrained');ax.bar(ledger.instrument,ledger.notional,color=['#39836b','#b65d5d']);ax.axhline(0,color='gray');ax.set(ylabel='Nominal con signo (EUR)',title='Cobertura nominal aproximada · precios hipotéticos');plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Una cobertura neutral sigue teniendo exposición bruta</b>

- Largo 0,1 BTC spot y corto 0,1 BTC perpetuo: exposición **neta −10 €** (casi cero) y **bruta 12.610 €**, un apalancamiento bruto de **1,8** sobre 7.000 € de patrimonio.
- El PnL conjunto es de −10 €: el spot gana 300 € y el perpetuo pierde 310 €, porque el perpetuo subió 100 € más (el basis de P07).
- La exposición neta mide el riesgo de dirección; la bruta mide cuánto se tiene en juego (y lo que exigen los márgenes y los costes).

</div>

## 9. Comprobaciones automáticas
Casos conocidos: alta bid +1, reducción ask +1, cancelación bid −2, desaparición ask +1, mejora bid +1,5 y mejora ask −2. El primer estado carece de OFI anterior.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: OFI conocido, OBI acotado, segmentos, frontera, exposición y resincronización
assert np.isnan(example.ofi_event.iloc[0])
assert np.allclose(example.ofi_event.iloc[1:], [1,1,-2,1,1.5,-2])
assert x.obi.between(-1,1).all()
assert x.loc[x.segment.ne(x.segment.shift()),'ofi_event'].isna().all()
assert x.loc[cut-1,'future_bps']!=x.loc[cut-1,'future_bps']
assert np.isclose(ledger.unrealized_pnl.sum(),-10)
assert np.isclose(exposures['gross_exposure'],12610)
assert np.isclose(exposures['net_exposure'],-10)
try:
    replay(snapshot,[updates[1]])
except ValueError: pass
else: raise AssertionError('No detectó salto de secuencia')
print('9 comprobaciones superadas')

## 10. Ejercicios
**E1. Reconstrucción.** Añade un evento que elimine un nivel no óptimo: ¿cambia OFI? ¿cambia la profundidad total? Añade después una mejora del ask.

**E2. OFI manual.** Implementa la fórmula sin vectorización, compara su resultado con `features`. Pista: usa dos filas consecutivas; el primer resultado debe ser NaN.

**E3. Ventanas.** Agrega OFI en ventanas de 5 y 15 segundos por segmento y compara con cambios de mid entre extremos. Excluye ventanas incompletas; identifica a qué instante se conoce cada variable.

**E4. Predicción.** Calcula el retorno futuro a 5 actualizaciones sin cruzar segmentos ni el corte temporal. ¿Se mantiene el signo y el R² de la relación? No optimices sobre validación.

**E5. Market Impact.** Compara la respuesta a 0, 5 y 30 segundos por cuartiles de tamaño. ¿Qué parte parece transitoria? ¿Por qué no podemos llamar permanente al último punto?

**E6. Exposición.** Construye una cartera con 3.000 EUR long BTC, 2.000 long ETH, 1.500 short SOL y 500 short XRP. Equity 5.000. Resultados esperados: neta 3.000, bruta 7.000, apalancamiento bruto 1,4. Diseña luego un escenario con neta cero y bruta positiva.

**E7. Datos reales.** Exporta cotizaciones reales desde P06, selecciona un exchange y ejecuta P13 en modo CSV. Registra origen, intervalo, cobertura y huecos. No llames OFI exacto al cálculo entre snapshots. Diseña un contrato de datos para capturar eventos L2 y trades con secuencia y timestamps en el módulo 2.

**E8. Interpretación.** ¿Qué diferencia hay entre explicar un cambio, anticiparlo y ganar dinero con él? Resume los resultados sin extrapolar los sintéticos al mercado real.

### E1 · Eliminar un nivel no óptimo y mejorar el ask

In [ ]:
updates_e1 = updates + [(7, 'bid', 98, 0., 'elimina nivel no óptimo'), (8, 'ask', 100.5, 1., 'mejora ask')]
e1 = features(replay(snapshot, updates_e1))
display(e1[['seq', 'event', 'bid', 'ask', 'bid_size', 'ask_size', 'bid_depth', 'ask_depth', 'ofi_event', 'mid_change_bps']].round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · El OFI de primer nivel no ve los niveles profundos</b>

| Evento | Mejor bid / ask | Profundidad bid | OFI | Cambio de mid |
|---|---|---|---|---|
| 7 · elimina el bid de 98 (3 BTC) | 100 / 101 | 5,5 → **2,5 BTC** | **0** | 0 bps |
| 8 · ask nuevo a 100,5 (1 BTC) | 100 / 100,5 | 2,5 BTC | **−1** | −24,9 bps |

- **Eliminar un nivel que no es el mejor no cambia el OFI**, aunque la profundidad compradora caiga más de la mitad (de 5,5 a 2,5 BTC). Para el OFI de primer nivel, el libro "no ha cambiado".
- Eso es una limitación real: la liquidez que sostiene el precio puede desaparecer sin que el OFI lo registre. Los OFI multinivel suman las contribuciones de varios niveles para capturarlo.
- **La mejora del ask** (un vendedor ofrece más barato) cuenta como presión vendedora: OFI = −q^a nuevo = **−1** BTC, y el mid baja 0,25 €.

</div>

### E2 · OFI sin vectorizar

In [ ]:
print(inspect.getsource(manual_ofi))

e2 = pd.DataFrame({'features': example.ofi_event, 'manual': manual_ofi_series(example)})
display(e2)
assert np.allclose(e2.manual, e2.features, equal_nan=True)
assert np.allclose(manual_ofi_series(x.iloc[:300]).iloc[1:], features(x.iloc[:300]).ofi_event.iloc[1:])
print("Coincide en el ejemplo y en las 300 primeras cotizaciones de la serie.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Misma fórmula, fila a fila</b>

- La versión con `if` explícitos compara cada estado con el anterior y suma o resta las cuatro piezas: bid que aparece, bid que desaparece, ask que aparece y ask que desaparece.
- Da exactamente lo mismo que la versión vectorizada en el ejemplo ([NaN, 1, 1, −2, 1, 1,5, −2]) y en las 300 primeras cotizaciones de la simulación.
- El primer resultado es **NaN**: sin estado anterior no hay cambio que medir. Ponerlo a 0 diría que "no hubo flujo", que no es lo mismo que "no se sabe".

</div>

### E3 · OFI en ventanas de 5 y 15 segundos

In [ ]:
print(inspect.getsource(window_ofi))

e3_rows = {}
for w in ['5s', '15s']:
    wo = window_ofi(x, w)
    fit = sm.OLS(wo.mid_change_bps, sm.add_constant(wo.ofi)).fit()
    e3_rows[w] = {'ventanas completas': len(wo), 'pendiente_bps_por_btc': fit.params['ofi'], 'r2': fit.rsquared,
                  'corr': wo.ofi.corr(wo.mid_change_bps)}
display(pd.DataFrame(e3_rows).T.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Agregar no elimina la relación mecánica</b>

| Ventana | Ventanas completas | Pendiente | R² | Correlación |
|---|---|---|---|---|
| 1 s (sección 6) | — | 0,25 bps/BTC | 0,61 | — |
| 5 s | 479 | 0,29 bps/BTC | **0,70** | 0,84 |
| 15 s | 159 | 0,28 bps/BTC | 0,66 | 0,81 |

- Solo se conservan ventanas **completas** (5 o 15 cotizaciones, todas con OFI y con el cierre de la ventana anterior disponible). Se descartan las que tocan un hueco o el principio del segmento.
- **¿Cuándo se conoce cada variable?** El OFI de la ventana y el cambio de mid entre extremos se conocen **a la vez**, al cierre de la ventana. Por eso esta relación es contemporánea: explica el cambio, pero no permite anticiparlo.
- Al agregar, el R² sube un poco (0,61 → 0,70): el ruido de los cambios de spread se compensa dentro de la ventana, pero la relación sigue siendo la misma relación mecánica.

</div>

### E4 · Retorno futuro a 5 actualizaciones

In [ ]:
x = add_future(x, 5, cut)
e4 = ofi_regressions(x, cut, {'futuro (1)': 'future_bps', 'futuro (5)': 'future_5_bps'})
display(e4.round(5))
assert x.loc[cut-5:cut-1, 'future_5_bps'].isna().all()  # ningún objetivo cruza el corte

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · A 5 actualizaciones no queda nada</b>

| Horizonte | Periodo | Pendiente | p-valor | R² |
|---|---|---|---|---|
| 1 actualización | Desarrollo | 0,047 | ≈ 0 | 0,021 |
| 1 actualización | Validación | 0,038 | 0,001 | 0,014 |
| 5 actualizaciones | Desarrollo | 0,035 | 0,094 | 0,002 |
| 5 actualizaciones | Validación | **0,002** | **0,96** | **0,000** |

- El signo se mantiene, pero en validación la pendiente es prácticamente 0 y el R² es nulo.
- Es lo esperado: la persistencia simulada solo dura **un paso** (el 20 % del shock anterior). A 5 actualizaciones, el poco efecto se diluye entre cuatro cambios independientes.
- Las 5 últimas filas antes del corte y las de cada final de segmento no tienen objetivo: su horizonte cruzaría la frontera o un hueco.

</div>

### E5 · Impacto por cuartiles de tamaño

In [ ]:
print(inspect.getsource(impact_by_quartile))

e5 = impact_by_quartile(events, (0, 5, 30))
display(e5.round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Casi tres cuartas partes del impacto son transitorias</b>

| Cuartil de tamaño | 0 s | 5 s | 30 s | Parte que revierte |
|---|---|---|---|---|
| Q1 (pequeño) | 0,70 bps | 0,31 | 0,28 | 60 % |
| Q2 | 2,23 bps | 1,11 | 0,65 | 71 % |
| Q3 | 3,36 bps | 1,64 | 0,91 | 73 % |
| Q4 (grande) | 4,63 bps | 2,39 | 1,32 | 71 % |

- **El impacto crece con el tamaño**: los trades grandes mueven el precio unas 7 veces más que los pequeños.
- **Parte transitoria:** en torno al **71–73 %** del impacto inicial ha revertido a los 30 s (en Q1, el ruido pesa más: 60 %). Es lo programado: q · (3 + 8 · e^{−h/4}), con un componente transitorio de 8 sobre un total inicial de 11 (73 %).
- **¿Por qué no llamar permanente al punto de 30 s?** Porque solo dice que a los 30 s todavía no ha revertido del todo. Puede seguir revirtiendo después, y en datos reales a 30 s se mezclan otros trades, noticias y el flujo de otros participantes. Un impacto "permanente" exige un horizonte en que la respuesta se estabilice y controles para todo lo demás que mueve el precio.

</div>

### E6 · Exposición de una cartera

In [ ]:
positions_e6 = pd.DataFrame({'instrument': ['BTC', 'ETH', 'SOL', 'XRP'], 'quantity': [3000, 2000, -1500, -500],
                             'entry_price': 1., 'current_price': 1.})  # nominales en EUR (precio 1)
_, exp_e6 = exposure(positions_e6, 5000)
assert (exp_e6['net_exposure'], exp_e6['gross_exposure'], exp_e6['gross_leverage']) == (3000, 7000, 1.4)
neutral = pd.DataFrame({'instrument': ['BTC spot', 'BTC perpetuo', 'ETH spot', 'ETH perpetuo'],
                        'quantity': [2000, -2000, 1000, -1000], 'entry_price': 1., 'current_price': 1.})
_, exp_neutral = exposure(neutral, 5000)
display(pd.DataFrame({'cartera E6': exp_e6, 'neta cero': exp_neutral}).round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Neta cero no es riesgo cero</b>

| | Cartera del ejercicio | Escenario neutral |
|---|---|---|
| Larga | 5.000 € | 3.000 € |
| Corta | 2.000 € | 3.000 € |
| **Neta** | **3.000 €** | **0 €** |
| **Bruta** | **7.000 €** | 6.000 € |
| Apalancamiento bruto | **1,4** | 1,2 |

- La cartera del ejercicio da exactamente lo pedido: neta 3.000, bruta 7.000 y apalancamiento 1,4 sobre 5.000 €.
- **Escenario con neta cero y bruta positiva:** largo 2.000 € de BTC spot y corto 2.000 € de BTC perpetuo, más largo 1.000 € de ETH spot y corto 1.000 € de ETH perpetuo. Neta 0, bruta 6.000 €.
- Con neta cero, un movimiento **común** del mercado no afecta al conjunto. Pero quedan el basis, el funding, el margen de cada perpetuo, la contraparte y la correlación imperfecta si las piernas no son el mismo activo (P07).

</div>

### E7 · Datos reales: cotizaciones de Kraken
No hay un CSV de P06 exportado; capturamos directamente las cotizaciones con la función de P06 (`collect_live`), que lee el mejor nivel de Kraken del libro `Depth`. Cambia `LIVE_E7` a `True` para capturar `LIVE_ROUNDS` cotizaciones (una por segundo).

In [ ]:
LIVE_E7 = False
LIVE_ROUNDS = 150

if LIVE_E7:
    from quantlab.p06.cross_exchange import collect_live
    live_raw, live_errors = collect_live(LIVE_ROUNDS, ('kraken',))  # los fallos quedan registrados
    if not live_errors.empty: display(live_errors)
    live = prepare_quotes(live_raw, 'kraken', SYMBOL, MAX_GAP_SECONDS)
    live_cut = int(len(live) * .65)
    live = add_future(live, 1, live_cut)
    print(f"{len(live)} cotizaciones | {live.received_at.min()} — {live.received_at.max()} | segmentos: {live.segment.nunique()} "
          f"| separación máxima: {live.gap_s.max():.3f} s | OFI nulo: {(live.ofi_event.fillna(0) == 0).mean():.1%}")
    print(f"Spread mediano: {((live.ask - live.bid) / ((live.ask + live.bid) / 2) * 1e4).median():.2f} bps | "
          f"tamaño mediano bid/ask: {live.bid_size.median():.2f} / {live.ask_size.median():.2f} BTC | "
          f"estados distintos: {len(live[['bid', 'ask', 'bid_size', 'ask_size']].drop_duplicates())} de {len(live)}")
    live_result = ofi_regressions(live, live_cut, {'contemporáneo': 'mid_change_bps', 'futuro': 'future_bps'})
    display(live_result.round(4))
    live_train = live_result[(live_result.target == 'futuro') & (live_result.period == 'desarrollo')]
    live_rule = sign_rule(live, live_cut, live_train.slope_bps_per_btc.iloc[0], 'future_bps', 20.)
    print('Regla de signo en validación:', {k: live_rule[k] for k in ('trades', 'gross_bps', 'net_bps', 'hit_rate')})
else:
    print("LIVE_E7 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Kraken BTC/EUR: relación contemporánea sí, predictiva no</b>

**Origen:** API REST pública de Kraken, endpoint `Depth` (mejor nivel), consultado una vez por segundo con la función de P06. **Intervalo:** 19-11-2026, de 21:24:35 a 21:27:04 UTC (150 cotizaciones, 0 errores). **Huecos:** ninguno; separación entre cotizaciones de 0,89 a 1,11 s (un solo segmento). Spread mediano: 0,28 bps; tamaño mediano del mejor nivel: 0,10 BTC (bid) y 0,24 BTC (ask).

| Objetivo | Periodo | n | Pendiente (bps por BTC) | p-valor | R² |
|---|---|---|---|---|---|
| Contemporáneo | Desarrollo | 96 | 4,02 | ≈ 0 | 0,50 |
| Contemporáneo | Validación | 53 | 5,27 | ≈ 0 | 0,56 |
| Futuro (t + 1) | Desarrollo | 95 | −0,04 | 0,76 | 0,00 |
| Futuro (t + 1) | Validación | 52 | 0,10 | 0,32 | 0,00 |

- **En el 56 % de las cotizaciones el OFI es nulo** (el mejor nivel no cambió de precio ni de tamaño; incluye la primera fila, sin OFI): el snapshot REST suele repetirse. Solo hubo 67 estados distintos en 150 consultas.
- **La relación contemporánea existe**, con una pendiente unas 20 veces mayor que en la simulación (los tamaños reales del mejor nivel son mucho menores). Pero es la misma relación casi mecánica de la sección 6.
- **No hay relación predictiva** (p = 0,76 y 0,32). La regla de signo hizo 24 operaciones en validación y ninguna ganó en bruto.
- **Esto no es OFI exacto:** entre dos snapshots separados por un segundo pueden ocurrir decenas de cambios del libro (altas, cancelaciones, ejecuciones) que se compensan y no se ven. Es un OFI **entre snapshots**, una aproximación.

**Contrato de datos para el módulo 2 (eventos L2 y trades):**

| Campo | Descripción |
|---|---|
| `exchange`, `symbol` | Un solo mercado por fichero |
| `channel` | `book` o `trade` (WebSocket) |
| `sequence` | Número de secuencia del exchange; un salto obliga a resincronizar |
| `checksum` | Si el exchange lo da, para verificar el libro reconstruido |
| `exchange_ts` | Hora del evento según el exchange (UTC, con su resolución) |
| `received_ts` | Hora de recepción local, con el reloj sincronizado por NTP y su desfase registrado |
| `side`, `price`, `size` | Para el libro: cantidad **absoluta** del nivel tras el evento (0 = eliminado) |
| `trade_id`, `taker_side`, `trade_price`, `trade_size` | Para trades: identificador único y lado agresor |
| `is_snapshot` | Marca los snapshots completos usados para (re)sincronizar |
| Metadatos | Versión del recolector, profundidad suscrita, periodos sin conexión y resincronizaciones |

Con este contrato se puede calcular el OFI exacto evento a evento, sincronizar cada trade con el libro de antes y de después y medir la respuesta del precio a trades reales (E5 con datos empíricos).

</div>

### E8 · Explicar, anticipar y ganar dinero

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Interpretación</b>

**Explicar un cambio** es relacionarlo con algo que ocurrió **a la vez**. El OFI explica el 61 % de la varianza del cambio de mid del mismo segundo en la simulación, y el 50–56 % en Kraken. Pero esa relación es en buena parte mecánica: el OFI se construye con los mismos movimientos del mejor bid y ask que mueven el mid. Sirve para entender la microestructura, no para operar.

**Anticiparlo** es relacionarlo con algo conocido **antes**. El OFI de t anticipa el cambio de t a t + 1 con un R² de 0,01–0,02 en la simulación (por una persistencia programada de un paso), nada a 5 actualizaciones y nada en Kraken.

**Ganar dinero con ello** exige que lo anticipado supere los costes. La regla acierta el 56 % de las veces en la simulación y gana 0,04 bps de media, frente a 20 bps de coste: **−19,96 bps** por operación. En Kraken, ninguna operación ganó en bruto.

**Resumen sin extrapolar:** los resultados sintéticos verifican el método (reconstrucción, OFI, segmentos, frontera), pero las relaciones que muestran están programadas. Con 2,5 minutos de cotizaciones REST de Kraken, la conclusión es que hay relación contemporánea entre snapshots y **evidencia insuficiente** de cualquier capacidad predictiva. Para ir más allá hace falta el contrato de datos del E7: eventos L2 y trades con secuencia y timestamps.

</div>

## 11. Exportación y conclusiones
El export es opcional. Las columnas de OFI y retorno futuro son conceptos diferentes; nunca utilices `future_bps` como feature de entrada. La lección siguiente, 1.28, profundizará en capitalización e interés compuesto. P14 ampliará el universo de activos.

In [ ]:
if EXPORT:
    folder=Path('quantlab_exports')/f'p13_{DATA_MODE}'
    folder.mkdir(parents=True,exist_ok=True)
    x.to_csv(folder/'ofi_features.csv',index=False)
    result.to_csv(folder/'impact_regressions.csv',index=False)
    print(folder.resolve())
else:
    print('Exportación desactivada.')

### Diario de investigación

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ DIARIO DE INVESTIGACIÓN · P13</b>

- **Fuente y periodo:** simulación reproducible (2.400 cotizaciones, semilla 73) y 150 cotizaciones reales de Kraken BTC/EUR (19-11-2026, 21:24–21:27 UTC, API REST `Depth`).
- **¿Qué aprendí sobre OFI?** Mide el **cambio** de la liquidez en el mejor nivel, no su estado (eso es el OBI). Su relación con el precio del mismo intervalo es alta pero en gran parte mecánica (correlación de 0,85 con el signo del cambio de mid), y su poder predictivo es mínimo: nulo a 5 actualizaciones y en Kraken. El OFI de primer nivel tampoco ve los cambios en niveles profundos (E1), y entre snapshots REST es solo una aproximación.
- **¿Qué aprendí sobre impacto y exposición?** El impacto crece con el tamaño y revierte en gran parte (≈ 71–73 % a los 30 s en el laboratorio), y un punto a 30 s no demuestra impacto permanente. Una exposición neta cero puede convivir con una exposición bruta alta y con riesgos que no son de dirección.
- **Limitaciones:** cotizaciones REST de un segundo (sin eventos entre medias), sin trades sincronizados, 2,5 minutos de datos reales y un experimento de impacto con parámetros inventados.
- **Siguiente hipótesis:** con eventos L2 y trades por WebSocket (contrato del E7), *el OFI multinivel agregado en 1 s anticipa el cambio del mid del segundo siguiente en BTC/EUR de Kraken más allá de su propia autocorrelación*, rechazable si en un periodo reservado de varias semanas el R² incremental no es positivo con errores HAC, o si la ganancia no supera el spread medido.

📄 Resumen en `docs/conclusiones/P13_order_flow_market_impact.md`

</div>